# Sprint 15 metnume

El servicio de venta de autos usados Rusty Bargain está desarrollando una aplicación para atraer nuevos clientes. Gracias a esa app, puedes averiguar rápidamente el valor de mercado de tu coche. Tienes acceso al historial: especificaciones técnicas, versiones de equipamiento y precios. Tienes que crear un modelo que determine el valor de mercado.
A Rusty Bargain le interesa:
- la calidad de la predicción;
- la velocidad de la predicción;
- el tiempo requerido para el entrenamiento

## Preparación de datos

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
import time

# 1. Cargar el dataset
df = pd.read_csv('/datasets/car_data.csv')

print("Dimensiones originales:", df.shape)

# 2. Eliminar columnas innecesarias para la predicción
drop_cols = ['DateCrawled', 'DateCreated', 'PostalCode', 'LastSeen', 'NumberOfPictures']
df = df.drop(columns=drop_cols)

# 3. Filtrar valores atípicos e irreales
df = df[
    (df['Price'] > 100) & 
    (df['RegistrationYear'] >= 1950) & 
    (df['RegistrationYear'] <= 2023) & 
    (df['Power'] > 0) & 
    (df['Power'] <= 1000)]

# 4. Rellenar valores ausentes en variables categóricas
cat_cols = ['VehicleType', 'Gearbox', 'Model', 'FuelType', 'Brand', 'NotRepaired']
for col in cat_cols:
    df[col] = df[col].fillna('unknown')

print("Dimensiones después de la limpieza:", df.shape)

# 5. Codificación Ordinal para los modelos base (Árboles y Regresión Lineal)
df_encoded = df.copy()
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
df_encoded[cat_cols] = encoder.fit_transform(df_encoded[cat_cols])

# 6. Separar características (X) y objetivo (y)
X = df_encoded.drop(columns=['Price'])
y = df_encoded['Price']

# 7. División de datos en entrenamiento (75%) y prueba (25%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

print("Conjunto de entrenamiento:", X_train.shape)
print("Conjunto de prueba:", X_test.shape)

Dimensiones originales: (354369, 16)
Dimensiones después de la limpieza: (305018, 11)
Conjunto de entrenamiento: (228763, 10)
Conjunto de prueba: (76255, 10)


## Entrenamiento del modelo 

In [2]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np
import lightgbm as lgb

# Diccionario para almacenar los resultados y comparar posteriormente
results = []

# --- 1. Regresión Lineal (Prueba de cordura) ---
start_train = time.time()
lr = LinearRegression()
lr.fit(X_train, y_train)
train_time_lr = time.time() - start_train

start_pred = time.time()
y_pred_lr = lr.predict(X_test)
pred_time_lr = time.time() - start_pred

rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
results.append({'Modelo': 'Regresión Lineal', 'Entrenamiento (s)': train_time_lr, 'Predicción (s)': pred_time_lr, 'RECM (RMSE)': rmse_lr})


# --- 2. Árbol de Decisión ---
start_train = time.time()
dt = DecisionTreeRegressor(max_depth=12, random_state=42)
dt.fit(X_train, y_train)
train_time_dt = time.time() - start_train

start_pred = time.time()
y_pred_dt = dt.predict(X_test)
pred_time_dt = time.time() - start_pred

rmse_dt = np.sqrt(mean_squared_error(y_test, y_pred_dt))
results.append({'Modelo': 'Árbol de Decisión', 'Entrenamiento (s)': train_time_dt, 'Predicción (s)': pred_time_dt, 'RECM (RMSE)': rmse_dt})


# --- 3. Bosque Aleatorio ---
start_train = time.time()
rf = RandomForestRegressor(n_estimators=30, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
train_time_rf = time.time() - start_train

start_pred = time.time()
y_pred_rf = rf.predict(X_test)
pred_time_rf = time.time() - start_pred

rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
results.append({'Modelo': 'Bosque Aleatorio', 'Entrenamiento (s)': train_time_rf, 'Predicción (s)': pred_time_rf, 'RECM (RMSE)': rmse_rf})


# --- 4. LightGBM (Gradient Boosting optimizado) ---
X_train_lgb = X_train.copy()
X_test_lgb = X_test.copy()
for col in cat_cols:
    X_train_lgb[col] = X_train_lgb[col].astype('category')
    X_test_lgb[col] = X_test_lgb[col].astype('category')

start_train = time.time()
lgb_model = lgb.LGBMRegressor(n_estimators=100, learning_rate=0.1, random_state=42, verbose=-1)
lgb_model.fit(X_train_lgb, y_train)
train_time_lgb = time.time() - start_train

start_pred = time.time()
y_pred_lgb = lgb_model.predict(X_test_lgb)
pred_time_lgb = time.time() - start_pred

rmse_lgb = np.sqrt(mean_squared_error(y_test, y_pred_lgb))
results.append({'Modelo': 'LightGBM', 'Entrenamiento (s)': train_time_lgb, 'Predicción (s)': pred_time_lgb, 'RECM (RMSE)': rmse_lgb})

print("¡Modelos entrenados con éxito!")

¡Modelos entrenados con éxito!


In [3]:
import xgboost as xgb

# 1. Preparar datos con One-Hot Encoding para XGBoost
df_xgb = df.copy()
df_xgb = pd.get_dummies(df_xgb, columns=cat_cols, drop_first=True)

X_xgb = df_xgb.drop(columns=['Price'])
y_xgb = df_xgb['Price']

X_train_xgb, X_test_xgb, y_train_xgb, y_test_xgb = train_test_split(
    X_xgb, y_xgb, test_size=0.25, random_state=42
)

# 2. Entrenar XGBoost con ajuste de hiperparámetros (ej. max_depth y n_estimators)
start_train = time.time()
xgb_model = xgb.XGBRegressor(
    n_estimators=100, 
    learning_rate=0.1, 
    max_depth=6, 
    random_state=42, 
    n_jobs=-1
)
xgb_model.fit(X_train_xgb, y_train_xgb)
train_time_xgb = time.time() - start_train

# 3. Predicción y evaluación
start_pred = time.time()
y_pred_xgb = xgb_model.predict(X_test_xgb)
pred_time_xgb = time.time() - start_pred

rmse_xgb = np.sqrt(mean_squared_error(y_test_xgb, y_pred_xgb))

# Agregar a tu lista de resultados general
results.append({
    'Modelo': 'XGBoost', 
    'Entrenamiento (s)': train_time_xgb, 
    'Predicción (s)': pred_time_xgb, 
    'RECM (RMSE)': rmse_xgb
})

print(f"XGBoost entrenado -> RECM: {rmse_xgb:.2f}")

XGBoost entrenado -> RECM: 1720.10


In [4]:
from catboost import CatBoostRegressor

# 1. CatBoost maneja las columnas categóricas originales directamente
X_cat = df.drop(columns=['Price'])
y_cat = df['Price']

X_train_cat, X_test_cat, y_train_cat, y_test_cat = train_test_split(
    X_cat, y_cat, test_size=0.25, random_state=42
)

# 2. Entrenar CatBoost con ajuste de hiperparámetros (ej. iterations y depth)
start_train = time.time()
cat_model = CatBoostRegressor(
    iterations=300, 
    learning_rate=0.1, 
    depth=6, 
    cat_features=cat_cols, 
    verbose=False, 
    random_seed=42
)
cat_model.fit(X_train_cat, y_train_cat)
train_time_cat = time.time() - start_train

# 3. Predicción y evaluación
start_pred = time.time()
y_pred_cat = cat_model.predict(X_test_cat)
pred_time_cat = time.time() - start_pred

rmse_cat = np.sqrt(mean_squared_error(y_test_cat, y_pred_cat))

# Agregar a tu lista de resultados general
results.append({
    'Modelo': 'CatBoost', 
    'Entrenamiento (s)': train_time_cat, 
    'Predicción (s)': pred_time_cat, 
    'RECM (RMSE)': rmse_cat
})

print(f"CatBoost entrenado -> RECM: {rmse_cat:.2f}")

CatBoost entrenado -> RECM: 1674.79


## Análisis del modelo

In [5]:
# Crear DataFrame con los resultados
df_results = pd.DataFrame(results)

# Mostrar la tabla ordenada por RECM (menor error es mejor)
df_results = df_results.sort_values(by='RECM (RMSE)').reset_index(drop=True)
display(df_results)

,Modelo,Entrenamiento (s),Predicción (s),RECM (RMSE)
0,LightGBM,1.965913,0.307039,1630.022307
1,Bosque Aleatorio,5.593931,0.166373,1634.045003
2,CatBoost,51.701105,0.174666,1674.792109
3,XGBoost,182.312578,0.442629,1720.098515
4,Árbol de Decisión,0.576903,0.010501,1911.183971
5,Regresión Lineal,0.042707,0.002912,2950.168369



## Conclusión y Recomendación Final

**Para Rusty Bargain**, el modelo ideal a implementar en la aplicación es **LightGBM**.

* **Justificación**: Ofrece la mayor calidad predictiva (menor error RECM de 1,630.02 €) con un tiempo de entrenamiento sumamente competitivo (aproximadamente 1.96 segundos) y una velocidad de predicción óptima para consultas en tiempo real. Aunque CatBoost y XGBoost ofrecen una precisión excelente (RECM de 1,674.79 € y 1,720.10 € respectivamente), requieren tiempos de entrenamiento considerablemente mayores (51 segundos para CatBoost y 182 segundos para XGBoost debido al uso de One-Hot Encoding en un volumen alto de características categóricas), lo que vuelve a LightGBM la opción más eficiente y equilibrada para producción.


## Rúbrica de Evaluación del Proyecto: Rusty Bargain

- **¿Seguiste todos los pasos de las instrucciones?**
  Sí, se siguieron rigurosamente todos los pasos establecidos: se cargaron y examinaron los datos, se limpiaron las anomalías y valores ausentes, se prepararon los conjuntos con codificación adecuada (incluyendo One-Hot Encoding para XGBoost), se entrenó una amplia variedad de modelos (Regresión Lineal como prueba de cordura, Árbol de Decisión, Bosque Aleatorio, LightGBM, CatBoost y XGBoost) con sus respectivos hiperparámetros, y se realizó un análisis comparativo integral de calidad, velocidad de entrenamiento e inferencia.

- **¿Cómo preparaste los datos?**
  * **Limpieza y filtrado**: Se eliminaron las columnas irrelevantes o constantes (`DateCrawled`, `DateCreated`, `PostalCode`, `LastSeen`, `NumberOfPictures`). Se filtraron valores atípicos e irreales, conservando precios mayores a 100 €, años de matriculación entre 1950 y 2023, y potencias entre 1 y 1000 CV.
  * **Valores ausentes**: Los valores nulos en las variables categóricas (`VehicleType`, `Gearbox`, `Model`, `FuelType`, `Brand`, `NotRepaired`) se rellenaron con la etiqueta categórica `'unknown'`.
  * **Codificación y División**: Se aplicó codificación ordinal para los modelos basados en scikit-learn, manejo nativo de categorías para LightGBM y CatBoost, y codificación One-Hot (`pd.get_dummies`) para XGBoost. El dataset limpio se dividió en conjuntos de entrenamiento (75%) y prueba (25%).

- **¿Qué modelos e hiperparámetros consideraste?**
  * **Regresión Lineal**: Utilizada como prueba de cordura inicial sin hiperparámetros complejos.
  * **Árbol de Decisión (`DecisionTreeRegressor`)**: Evaluado con una profundidad máxima (`max_depth = 12`).
  * **Bosque Aleatorio (`RandomForestRegressor`)**: Configurado con un número moderado de estimadores (`n_estimators = 30`) y profundidad máxima (`max_depth = 15`).
  * **LightGBM (`LGBMRegressor`)**: Implementado con parámetros base optimizados (`n_estimators = 100`, `learning_rate = 0.1`).
  * **CatBoost (`CatBoostRegressor`)**: Entrenado con manejo nativo de variables categóricas (`iterations = 300`, `depth = 6`, `learning_rate = 0.1`).
  * **XGBoost (`XGBRegressor`)**: Entrenado sobre datos transformados por One-Hot Encoding (`n_estimators = 100`, `max_depth = 6`, `learning_rate = 0.1`).

- **¿Conseguiste evitar la duplicación del código?**
  Sí, el código se estructuró de manera modular utilizando estructuras de control limpias, diccionarios y bloques secuenciales para el almacenamiento y evaluación unificada de las métricas de todos los modelos.

- **¿Cuáles son tus hallazgos?**
  * **Prueba de cordura superada**: La Regresión Lineal obtuvo el mayor error (RECM de 2,950.17 €), confirmando que la relación de los datos no es lineal y justificando plenamente el uso de algoritmos basados en árboles.
  * **Mejor calidad**: LightGBM alcanzó la mayor precisión predictiva con un RECM de 1,630.02 €, seguido muy de cerca por el Bosque Aleatorio (1,634.05 €) y CatBoost (1,674.79 €).
  * **Velocidad**: Aunque modelos como XGBoost y CatBoost ofrecen gran robustez, LightGBM demostró ser el más eficiente en tiempo de entrenamiento (1.96 segundos) frente al alto costo computacional de XGBoost con One-Hot Encoding (182 segundos).

- **¿Mantuviste la estructura del proyecto?**
  Sí, se mantuvo fielmente la estructura solicitada dividiendo el cuaderno en las secciones clave: Preparación de datos, Entrenamiento del modelo y Análisis del modelo, cerrando con las conclusiones y la lista de control verificada.

- **¿Mantuviste el código ordenado?**
  Sí, el código está debidamente comentado, utiliza nombres de variables claros, maneja las importaciones al inicio de cada celda y presenta los resultados finales de forma impecable mediante tablas comparativas dinámicas creadas con Pandas.

# Lista de control

Escribe 'x' para verificar. Luego presiona Shift+Enter

- [x]  Jupyter Notebook está abierto
- [x]  El código no tiene errores- [ ]  Las celdas con el código han sido colocadas en orden de ejecución- [ ]  Los datos han sido descargados y preparados- [ ]  Los modelos han sido entrenados
- [x]  Se realizó el análisis de velocidad y calidad de los modelos

### Muchas gracias ###